In [ ]:
from transformers import pipeline
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, f_oneway
from sklearn.cluster import KMeans, OPTICS, DBSCAN, AgglomerativeClustering, SpectralClustering, AffinityPropagation, MeanShift
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, PCA, LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.mixture import GaussianMixture
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import statsmodels.api as sm
from sentence_transformers import SentenceTransformer
from scipy import stats
from textblob import TextBlob
from nltk.tokenize import word_tokenize
import nltk
import numpy as np
nltk.download('punkt')

In [ ]:
# model = SentenceTransformer("all-MiniLM-L6-v2")
# combined_e1["embedding"] = combined_e1["Sentences Overall"].apply(model.encode)
# combined_e2["embedding"] = combined_e2["Sentences Overall"].apply(model.encode)
# combined_e1.to_pickle('./text/combined_e1_rob.pkl')
# combined_e2.to_pickle('./text/combined_e2_rob.pkl')

In [ ]:
# Added a sentiment
combined_e1 = pd.read_pickle('./text/combined_e1_rob.pkl')
combined_e2 = pd.read_pickle('./text/combined_e2_rob.pkl')

Rating-based clustering

In [ ]:
features_for_clustering_e1 = combined_e1[
    ["Interesting"]
].dropna()

features_for_clustering_e2 = combined_e2[
    ["Interesting"]
].dropna()

def cluster_features(features, df):
    """
    Applies K means clustering with K=3 to the group datapoints by a given feature
    """
    new_df = df.copy()
    for feature in features:
        features_for_clustering = new_df[[feature]]
        kmeans = KMeans(n_clusters=3, random_state=0).fit(features_for_clustering)
        new_df["Cluster_" + feature] = kmeans.labels_
    return new_df


new_df_1 = cluster_features(
    ["Interesting", "Pleasant", "Engaging", "Enjoy Working"], combined_e1
)
new_df_2 = cluster_features(
    ["Interesting", "Pleasant", "Engaging", "Enjoy Working"], combined_e2
)

In [ ]:
def anova_for_groups(new_df_2, label):
    """ 
    Applies one-way ANOVA test to check if the clusters have significant differences in their average values 
    """
    print(label)
    # Group the data by cluster labels
    groups = [new_df_2['Interesting'][new_df_2['Cluster_Interesting'] == label] for label in new_df_2['Cluster_Interesting'].unique()]
    # Perform ANOVA test
    f_statistic, p_value = stats.f_oneway(*groups)

    # Print the results
    print("F-statistic:", f_statistic)
    print("P-value:", p_value)

    # Determine significance based on the p-value
    alpha = 0.05  # Set your desired significance level (e.g., 0.05)
    if p_value < alpha:
        print("The means within clusters are significantly different.")
    else:
        print("There is no significant difference in means within clusters.")

anova_for_groups(new_df_1, "Experiment 1")
anova_for_groups(new_df_2, "Experiment 2")

In [ ]:
# Check the distribution of scene number or scene type in the identified clusters
def my_anova_test(new_df, feature, dep):
    contingency_table = pd.crosstab(new_df[f"Cluster_{feature}"], new_df[dep])

    # Perform the chi-square test
    chi2, p, _, _ = chi2_contingency(contingency_table)
    num_comparisons = len(pd.unique(new_df[dep]))

    # Check for significance
    alpha = 0.05
    # Apply bonferroni correction
    bonferroni_alpha = alpha / num_comparisons
    print(f"p value: {p}")
    print("\n")
    if p < bonferroni_alpha:
        print(
            f"There are significant differences in {dep} within the clusters based on {feature}"
        )
    else: print(
            f"There are no significant differences in {dep} within the clusters based on {feature}"
        )
    return contingency_table
contingency_table1 = my_anova_test(new_df_1, "Interesting", "Scene Number")
contingency_table2= my_anova_test(new_df_2, "Interesting", "Scene Number")

In [ ]:
def find_dominant_scenes_and_clusters(contingency_table, factor=2):
    """
    Find scene numbers where the dominant cluster has more than 'factor' times 
    the frequency of any other cluster and identify the dominant cluster.

    Parameters:
    contingency_table (pd.DataFrame): The contingency table DataFrame.
    factor (float): The factor by which the dominant cluster frequency must exceed others. Default is 2.

    Returns:
    list of tuples: Each tuple contains a scene number and its dominant cluster.
    """
    # Initialize an empty list to store the result
    dominant_scenes_and_clusters = []

    # Iterate over each column (scene number)
    for scene_number in contingency_table.columns:
        # Get the values for the current scene number
        values = contingency_table[scene_number]
        
        # Sort the values in descending order and get the indices
        sorted_values = values.sort_values(ascending=False)
        
        # Check if the highest value is more than 'factor' times the second highest value
        if sorted_values.iloc[0] > factor * sorted_values.iloc[1]:
            # Get the index (cluster) of the highest value
            dominant_cluster = sorted_values.index[0]
            # Append the scene number and dominant cluster to the result list
            dominant_scenes_and_clusters.append((scene_number, dominant_cluster))

    return dominant_scenes_and_clusters

# Find scene numbers with a dominant cluster with a factor of 2
dominant_scenes = find_dominant_scenes_and_clusters(contingency_table1, factor=2)
print("Exp1: Scene numbers with a dominant cluster:", dominant_scenes)

dominant_scenes = find_dominant_scenes_and_clusters(contingency_table2, factor=2)
print("Exp2: Scene numbers with a dominant cluster:", dominant_scenes)


In [ ]:
cont1 = my_anova_test(new_df_1, "Interesting", "Scene Type")
# Find scene types with a dominant cluster with a factor of 2
dominant_scenes = find_dominant_scenes_and_clusters(cont1, factor=2)
print("Exp1: Scene types with a dominant cluster:", dominant_scenes)

cont2 = my_anova_test(new_df_2, "Interesting", "Scene Type")
# Find scene numbers with a dominant cluster with a factor of 2
dominant_scenes = find_dominant_scenes_and_clusters(cont2, factor=2)
print("Exp2: Scene types with a dominant cluster:", dominant_scenes)

In [ ]:
def most_vs_least_popular(contingency_table):
    """
    For each row in the contingency table, print how much the most popular type 
    is larger than the least popular type.

    Parameters:
    contingency_table (pd.DataFrame): The contingency table DataFrame.
    """
    # Iterate over each row in the contingency table
    for index, row in contingency_table.iterrows():
        # Find the most popular and least popular types
        most_popular = row.max()
        least_popular = row.min()
        
        # Calculate the difference
        difference = (most_popular / least_popular - 1) * 100
        
        # Print the result
        print(f"{index}: Most popular type is larger than least popular type by {int(difference)}%")
most_vs_least_popular(cont2)

In [ ]:
def bar_plot_clusters(new_df_2, feature, label):
    """ 
    Creates bar plots for clusters based on the provided feature
    """
    # Assuming "df" is your DataFrame
    plt.figure(figsize=(5, 5))  # Set the figure size as needed

    # Create the boxplot based on "labels" and "interesting" feature
    sns.boxplot(data=new_df_2, x=f'Cluster_{feature}', y='Interesting')

    # Set labels for the plot
    plt.xlabel('Cluster Labels')
    plt.ylabel(f'{feature} Feature')

    # Set a title for the plot
    plt.title(label)

    # Show the plot
    plt.savefig(f'figures/{label}_{feature}.png')
    plt.show()

bar_plot_clusters(new_df_1, 'Interesting', 'Experiment 1')
bar_plot_clusters(new_df_2, 'Interesting', 'Experiment 2')

Text-based clustering

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
count = CountVectorizer(stop_words="english")

def top_words(df, n=10):
    text_data_vectorized = vectorizer.fit_transform(df["Sentences Overall"])
    # Sum the TF-IDF values for each word
    tfidf_sum = np.sum(text_data_vectorized, axis=0).A1

    # Get the words (feature names)
    words = vectorizer.get_feature_names_out()

    # Sort the words by their summed TF-IDF values in descending order
    sorted_indices = np.argsort(tfidf_sum).flatten()[::-1]
    sorted_words = [words[i] for i in sorted_indices]
    return sorted_words[:n]

def unique_words_per_cluster(new_df_1):
        aggregated_df_1 = new_df_1.groupby('Cluster_Interesting')['Sentences Overall'].apply(' '.join).reset_index()
        cluster_top_words = {}
        for index, row in aggregated_df_1.iterrows():
                cluster_id = row['Cluster_Interesting']
                sentences = pd.DataFrame({'Sentences Overall': [row['Sentences Overall']]})
                top_words_list = top_words(sentences)
                cluster_top_words[cluster_id] = set(top_words_list)

        # Print unique words for each cluster by computing the difference
        for cluster_id, words in cluster_top_words.items():
                other_clusters = set.union(*[words for cid, words in cluster_top_words.items() if cid != cluster_id])
                unique_words = words - other_clusters
                print(f"Cluster {cluster_id}: {unique_words}")

print(f"Experiment 1:\n")
unique_words_per_cluster(new_df_1)
print(f"Experiment 2:\n")
unique_words_per_cluster(new_df_2)

In [ ]:
def get_top_n_tfidf_features(corpus, n=50):
    """
    Transforms the given corpus into TF-IDF embeddings and returns the top n features by TF-IDF score.

    Parameters:
    corpus (list of str): The corpus of documents to transform.
    n (int): The number of top features to keep.

    Returns:
    np.ndarray: The TF-IDF embedding matrix with only the top n features.
    """
    vectorizer = TfidfVectorizer()
    tfid_embedding = vectorizer.fit_transform(corpus)
    tfid_embedding = tfid_embedding.toarray()
    
    # Sum up the TF-IDF values for each word
    tfidf_sums = tfid_embedding.sum(axis=0)
    
    # Get the indices of the top n words
    top_n_indices = np.argsort(tfidf_sums)[-n:]
    
    # Sort the indices to keep the columns in the same order
    top_n_indices = np.sort(top_n_indices)
    
    # Keep only the columns corresponding to the top n words
    top_n_tfidf_embedding = tfid_embedding[:, top_n_indices]
    
    return top_n_tfidf_embedding

def scale(data):
    """Applies scaling to data"""
    scaler = StandardScaler()
    s = scaler.fit_transform(data)
    return s

def plot_clusters(scaled_data, labels, title):
    """Projects clusters into 2D for visualization"""
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(scaled_data)

    plt.figure()
    plt.scatter(
        pca_result[:, 0],
        pca_result[:, 1],
        c=labels,
        cmap="viridis",
        edgecolors="k",
        s=50,
    )
    plt.xlabel('Principal Component 1')  # Label for x-axis
    plt.ylabel('Principal Component 2')  # Label for y-axis
    plt.title(title)
    plt.savefig(f'figures/{title}.png')
    plt.show()

def cluster(scaled_data, extra_labels, plot=True):
    """Applies various clustering methods to the scaled data"""
    # K-means clustering
    kmeans = KMeans(n_clusters=3, random_state=42)  # You can adjust the number of clusters as needed
    kmeans_labels = kmeans.fit_predict(scaled_data)

    # Hierarchical clustering
    agg_clustering = AgglomerativeClustering(n_clusters=5, linkage="ward")  # You can adjust the number of clusters and linkage as needed
    agg_labels = agg_clustering.fit_predict(scaled_data)

    # DBSCAN clustering
    dbscan = DBSCAN(metric="manhattan")
    dbscan_labels = dbscan.fit_predict(scaled_data)

    # Gaussian Mixture Model clustering
    gmm = GaussianMixture(n_components=3, random_state=42)  # Adjust the number of components (clusters) as needed
    gmm_labels = gmm.fit_predict(scaled_data)

    # Spectral clustering
    spectral = SpectralClustering(n_clusters=3, affinity='nearest_neighbors', random_state=42)
    spectral_labels = spectral.fit_predict(scaled_data)

    # Affinity Propagation clustering
    affinity = AffinityPropagation(random_state=42)
    affinity_labels = affinity.fit_predict(scaled_data)

    # Mean Shift clustering
    mean_shift = MeanShift()
    mean_shift_labels = mean_shift.fit_predict(scaled_data)
    label_encoder = LabelEncoder()

    if plot:
        plot_clusters(scaled_data, kmeans_labels, "K-means")
        plot_clusters(scaled_data, agg_labels, "Hierarchical")
        plot_clusters(scaled_data, dbscan_labels, "DBSCAN")
        plot_clusters(scaled_data, gmm_labels, "GMM")
        plot_clusters(scaled_data, spectral_labels, "Spectral")
        plot_clusters(scaled_data, affinity_labels, "Affinity Propagation")
        plot_clusters(scaled_data, mean_shift_labels, "Mean Shift")
        plot_clusters(scaled_data, label_encoder.fit_transform(extra_labels), "Custom Labels")

    return kmeans_labels, agg_labels, dbscan_labels, gmm_labels, spectral_labels, affinity_labels, mean_shift_labels

def to_dict(clusters):
    d = {}
    for e in clusters:
        if e in d: d[e] +=1
        else: d[e] = 1
    tot = sum(d.values())
    for e in d:
        d[e] = d[e] / tot * 100
    print(d)
    # return d

def reduce_dimension_with_lda(corpus, vectorizer, n_topics=50):
    """
    Reduces the dimensionality of the given corpus using LDA on the TF-IDF matrix.

    Parameters:
    corpus (list of str): The corpus of documents to transform.
    n_topics (int): The number of topics to reduce to.

    Returns:
    np.ndarray: The LDA embedding matrix with the given number of topics.
    """
    # Step 1: Transform the corpus into a TF-IDF matrix
    tfid_embedding = vectorizer.fit_transform(corpus)
    
    # Step 2: Fit the LDA model on the TF-IDF matrix
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=0)
    lda_embedding = lda.fit_transform(tfid_embedding)
    
    return lda_embedding

def vectorize_with_lda(corpus, n_topics=10):
    """
    Vectorizes the given corpus using LDA and returns the topic distribution matrix.

    Parameters:
    corpus (list of str): The corpus of documents to vectorize.
    n_topics (int): The number of topics to extract.

    Returns:
    np.ndarray: The LDA topic distribution matrix for the corpus.
    LatentDirichletAllocation: The fitted LDA model.
    CountVectorizer: The fitted CountVectorizer.
    """
    # Step 1: Convert the corpus into a document-term matrix
    vectorizer = CountVectorizer()
    doc_term_matrix = vectorizer.fit_transform(corpus)
    
    # Step 2: Fit the LDA model on the document-term matrix
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda_topic_matrix = lda.fit_transform(doc_term_matrix)
    
    return lda_topic_matrix, lda, vectorizer

In [ ]:
count_e1 = count.fit_transform(combined_e1["Sentences Overall"].values).toarray()
count_e2 = count.fit_transform(combined_e2["Sentences Overall"].values).toarray()
k_e1, h_e1, d, g, s, a, m = cluster(scale(count_e1), combined_e1["Scene Type"].values)

In [ ]:
def print_top_words_per_cluster(vectorizer, corpus, labels, n_clusters, n_top_words=10):
    """Prints the top words for each cluster"""
    count_matrix = vectorizer.fit_transform(corpus)
    feature_names = vectorizer.get_feature_names_out()

    for cluster_num in range(n_clusters):
        print(f"Cluster #{cluster_num + 1}:")
        cluster_docs = np.where(labels == cluster_num)[0]
        cluster_matrix = count_matrix[cluster_docs]
        word_counts = np.asarray(cluster_matrix.sum(axis=0)).flatten()
        top_word_indices = word_counts.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_word_indices]
        print(" ".join(top_words))
        print("\n")

# Print top words for K-means clusters
print_top_words_per_cluster(vectorizer, combined_e1["Sentences Overall"].values, k_e1, n_clusters=3)

In [ ]:
def contingency(combined_e1, k_e1, dep, title):
    # Crosstabulation of the data
    contingency_table = pd.crosstab(k_e1, combined_e1[dep])

    # Conduct Pearson chi-squared test
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

    # Display results
    print(f"{title}; dependent variable: {dep}")
    print(f"Chi-squared: {chi2}")
    print(f"P-value: {p}")

contingency(combined_e1, k_e1, 'Scene Type', 'Experiment 1')
contingency(combined_e1, k_e1, 'Scene Number', 'Experiment 1')

In [ ]:
lda_embedding2 = reduce_dimension_with_lda(combined_e2["Sentences Overall"], vectorizer, n_topics=50)
k_e2, h_e2, d2, g2, s2, a2, m2 = cluster(lda_embedding2, combined_e2['Scene Type'].values)

In [ ]:
contingency(combined_e2, s2, 'Scene Type', 'Experiment 2')
contingency(combined_e2, s2, 'Scene Number', 'Experiment 2')# Crosstabulation of the data
print_top_words_per_cluster(vectorizer, combined_e2["Sentences Overall"].values, k_e2, n_clusters=3)
to_dict(h_e1)
to_dict(h_e2)

In [ ]:
def diff(list1, list2):
    set1 = set(list1)
    set2 = set(list2)
    d = set2 - set1
    print(d)
    return d

mask2 = [e != 0 for e in h_e2]
outliers_e2 = combined_e2.loc[mask2]
normal_e2 = combined_e2.loc[[not e for e in mask2]]

mask = [e != 0 for e in h_e1]
outliers_e1 = combined_e1.loc[mask]
normal_e1 = combined_e1.loc[[not e for e in mask]]

d1 = diff(top_words(normal_e1), top_words(outliers_e1))
d2 = diff(top_words(normal_e2), top_words(outliers_e2))

In [ ]:
aggregated_type_e1 = combined_e1.groupby('Scene Type')['Sentences Overall'].apply(' '.join).reset_index()
aggregated_type_e2 = combined_e2.groupby('Scene Type')['Sentences Overall'].apply(' '.join).reset_index()
print(f"Common words from image class:")
top_words(aggregated_type_e1[aggregated_type_e1['Scene Type'] == 'Image'])
print(f"Common words from video class:")
top_words(aggregated_type_e1[aggregated_type_e1['Scene Type'] == 'Video'])

In [ ]:
# top words for each topic
def display_topics(model, feature_names, no_top_words):
    """Discovers topics using LDA model"""
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        topics["Topic %d:" % (topic_idx)] = " ".join(
            [feature_names[i] for i in topic.argsort()[: -no_top_words - 1 : -1]]
        )
    return topics

def top_words_lda(n_topics=1, n_top_words=10, all_text=None, vectorizer=None):
    text_data_vectorized = vectorizer.fit_transform(all_text['Sentences Overall'])

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        max_iter=5,
        learning_method="online",
        learning_offset=50.0,
        random_state=0,
    )
    lda.fit(text_data_vectorized)

    # feature names from the vectorizer
    tf_feature_names = vectorizer.get_feature_names_out()

    return display_topics(lda, tf_feature_names, n_top_words)


def analyze_scene_type_difference(df, dependent_cols, independent_col):
    results = {}
    
    for col in dependent_cols:
        group_means = df.groupby(independent_col)[col].mean().to_dict()
        results[col] = {'group_means': group_means}
        
        if col == 'Sentiment':
            # Perform Chi-Square test for categorical data
            contingency_table = pd.crosstab(df[col], df[independent_col])
            chi2, p, dof, ex = stats.chi2_contingency(contingency_table)
            results[col].update({'chi2': chi2, 'p-value': p, 'degrees of freedom': dof, 'expected frequencies': ex})
        else:
            unique_groups = df[independent_col].unique()
            if len(unique_groups) > 2:
                # Perform ANOVA
                formula = f'{col} ~ C({independent_col})'
                model = ols(formula, data=df).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                results[col].update(anova_table.to_dict())
            else:
                # Perform t-test
                group1 = df[df[independent_col] == unique_groups[0]][col]
                group2 = df[df[independent_col] == unique_groups[1]][col]
                t_stat, p_val = stats.ttest_ind(group1, group2)
                results[col].update({'t-statistic': t_stat, 'p-value': p_val})
        
        # Plotting boxplot
        if col != 'Sentiment':
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=independent_col, y=col, data=df)
            plt.title(f'Boxplot of {col} by {independent_col}')
            plt.show()
    
    return results

In [ ]:
df = combined_e2
dependent_columns = ['Pleasant', 'Interesting', 'Engaging', 'Enjoy Working', 'Sentiment']
independent_column = 'Scene Type'
results = analyze_scene_type_difference(df, dependent_columns, independent_column)
print(results)

In [ ]:
df = combined_e1
dependent_columns = ['Pleasant', 'Interesting', 'Engaging', 'Enjoy Working', 'Sentiment']
independent_column = 'Scene Type'
results = analyze_scene_type_difference(df, dependent_columns, independent_column)
print(results)

In [ ]:
def find_top_extreme_words(text, top_n=10):
    # Tokenize the text into words
    words = word_tokenize(text)
    
    # Initialize a dictionary to store sentiment scores
    word_sentiments = {}
    
    for word in words:
        # Not having the context ~ can be verified with manual check
        # Get the sentiment polarity of the word
        sentiment = TextBlob(word).sentiment.polarity
        word_sentiments[word] = sentiment
    
    # Sort the words by their sentiment scores
    sorted_words = sorted(word_sentiments.items(), key=lambda item: item[1])
    
    # Get the top n most negative and top n most positive words
    most_negative_words = sorted_words[:top_n]
    most_positive_words = sorted_words[-top_n:]
    
    # Convert lists to dictionaries for output
    most_negative_words_dict = {word: score for word, score in most_negative_words}
    most_positive_words_dict = {word: score for word, score in most_positive_words}
    
    # Return the results in a dictionary
    return {
        'most_positive_words': most_positive_words_dict,
        'most_negative_words': most_negative_words_dict
    }

r1 = find_top_extreme_words(aggregated_type_e2[aggregated_type_e2['Scene Type'] == 'Clear']['Sentences Overall'].values[0], top_n=50)
r2 = find_top_extreme_words(aggregated_type_e2[aggregated_type_e2['Scene Type'] == 'Overcast']['Sentences Overall'].values[0], top_n=50)

# Find unique positive words
unique_positive_r1 = {word: score for word, score in r1['most_positive_words'].items() if word not in r2['most_positive_words']}
unique_positive_r2 = {word: score for word, score in r2['most_positive_words'].items() if word not in r1['most_positive_words']}

# Find unique negative words
unique_negative_r1 = {word: score for word, score in r1['most_negative_words'].items() if word not in r2['most_negative_words']}
unique_negative_r2 = {word: score for word, score in r2['most_negative_words'].items() if word not in r1['most_negative_words']}

print("Unique positive words in Clear:", unique_positive_r1)
print("Unique positive words in Overcast:", unique_positive_r2)
print("Unique negative words in Clear:", unique_negative_r1)
print("Unique negative words in Overcast:", unique_negative_r2)


In [ ]:
r1 = find_top_extreme_words(aggregated_type_e1[aggregated_type_e1['Scene Type'] == 'Image']['Sentences Overall'].values[0], top_n=50)
r2 = find_top_extreme_words(aggregated_type_e1[aggregated_type_e1['Scene Type'] == 'Video']['Sentences Overall'].values[0], top_n=50)

# Find unique positive words
unique_positive_r1 = {word: score for word, score in r1['most_positive_words'].items() if word not in r2['most_positive_words']}
unique_positive_r2 = {word: score for word, score in r2['most_positive_words'].items() if word not in r1['most_positive_words']}

# Find unique negative words
unique_negative_r1 = {word: score for word, score in r1['most_negative_words'].items() if word not in r2['most_negative_words']}
unique_negative_r2 = {word: score for word, score in r2['most_negative_words'].items() if word not in r1['most_negative_words']}

print("Unique positive words in Image:", unique_positive_r1)
print("Unique positive words in Video:", unique_positive_r2)
print("Unique negative words in Image:", unique_negative_r1)
def print_top_words(model, feature_names, n_top_words):
    """
    Print the top words for each topic.
    
    Parameters:
    model (LatentDirichletAllocation): Fitted LDA model.
    feature_names (array): Array of feature names from the vectorizer.
    n_top_words (int): Number of top words to print for each topic.
    """
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic #{topic_idx + 1}:")
        print(" ".join([feature_names[i]
                        for i in topic.argsort()[:-n_top_words - 1:-1]]))
        print()

def set_dom(data, vectorizer):
    lda = LatentDirichletAllocation(n_components=5, random_state=42)
    
    # Fit the LDA model
    lda.fit(vectorizer.transform(data["Sentences Overall"]))
    
    # Transform the data
    lda_topics = lda.transform(vectorizer.transform(data["Sentences Overall"]))
    
    # Assign dominant topic
    data["Dominant_Topic"] = np.argmax(lda_topics, axis=1) + 1
    
    # Print top words for each topic
    feature_names = vectorizer.get_feature_names_out()
    print_top_words(lda, feature_names, 10)
    print("Unique negative words in Video:", unique_negative_r2)


print("Experiment 1\n")
set_dom(combined_e1, vectorizer)
print("Experiment 2\n")
set_dom(combined_e2, vectorizer)

In [ ]:
df = combined_e1.copy()
# Rename the columns for consistency
df.rename(columns={
    'Enjoy Working': 'Enjoy_Working',
    'Scene Type': 'Scene_Type',
    'Scene Number': 'Scene_Number'
}, inplace=True)

# Convert specified columns to categorical
df['Dominant_Topic'] = df['Dominant_Topic'].astype('category')
df['Scene_Type'] = df['Scene_Type'].astype('category')
df['Scene_Number'] = df['Scene_Number'].astype('category')

def nested_model_selection(df):
    """ 
    Identifies most relevant predictors using nested model selection with ANOVA
    """
    models = []
    formulas = [
        'Sentiment ~ Pleasant',
        'Sentiment ~ Pleasant + Enjoy_Working',
        'Sentiment ~ Pleasant + Enjoy_Working + Interesting',
        'Sentiment ~ Pleasant + Enjoy_Working + Dominant_Topic',
        'Sentiment ~ Pleasant + Enjoy_Working + Engaging',
        'Sentiment ~ Pleasant + Enjoy_Working + Engaging + Dominant_Topic'
    ]
    
    for formula in formulas:
        model = ols(formula, data=df).fit()
        models.append((formula, model))
    
    best_model = None
    best_anova = None
    min_p_value = float('inf')
    
    for i in range(len(models) - 1):
        anova_results = anova_lm(models[i][1], models[i+1][1])
        p_value = anova_results['Pr(>F)'][1]
        if p_value < min_p_value:
            min_p_value = p_value
            best_model = models[i+1]
            best_anova = anova_results
    
    # Print the summary of the best model
    print("Best Model Formula:", best_model[0])
    print(best_model[1].summary())
    print("\nANOVA Results:\n", best_anova)

# Call the nested_model_selection function on the combined dataframe
nested_model_selection(df)

In [ ]:
def run_regression_and_print_summary(df, formula):
    """
    Regress Sentiment on Scene_Type, Scene_Number, and Dominant_Topic.
    Print the model summary including coefficient significance.
    """
    model_formula = formula
    model = ols(model_formula, data=df).fit()
    
    # Print the summary of the model
    print(model.summary())

# Call the function on the combined dataframe
run_regression_and_print_summary(df, 'Sentiment ~ C(Scene_Type) + C(Scene_Number) + C(Dominant_Topic)')

In [ ]:
df2 = combined_e2.copy()
# Rename the columns for consistency and to remove spaces
df2.rename(columns={
    'Enjoy Working': 'Enjoy_Working',
    'Scene Type': 'Scene_Type',
    'Scene Number': 'Scene_Number'
}, inplace=True)

# Convert specified columns to categorical
df2['Dominant_Topic'] = df2['Dominant_Topic'].astype('category')
df2['Scene_Type'] = df2['Scene_Type'].astype('category')
df2['Scene_Number'] = df2['Scene_Number'].astype('category')

# Call the function on the combined dataframe
run_regression_and_print_summary(df2, 'Sentiment ~ C(Scene_Type) + C(Scene_Number) + C(Dominant_Topic)')

In [ ]:
# Call the function on the combined dataframe
run_regression_and_print_summary(df2, 'Sentiment ~ C(Scene_Type) + C(Scene_Number)')

In [ ]:
# Linear regression just with scene number
# Extract the top 10 words for each scene
# Cover both experiment 1 and 2
# Start with methodology
# The same methodology applied to 2 experiments
# From exp 1 we go from global to local
# Local level: outliers
# Run lda + tfidf for the clusters
# Frequency of the word in the extreme sentiment + how many participants said this word
# Group / cluster scene numbers based on topics and top words
# See where the extreme words come from (which scenes they are associated with)
# Visually refer to text
# Topics for clustering or word embeddings for clustering
# Bored vs boring
# The coeffients need to be related to words
# Group scenes with the highest significant coefficients and pull the topics from this group
# Scenes associated with negative coefficients and pull up the words

In [ ]:
# EPFL: present your thesis: hypothesis, methodology, results
# Questions designed to see your response to criticism
# Justify the methodology
# Be open to criticism